Phase 6 -- Train GCN and GAT with early stopping, evaluate with
imbalance-aware metrics, and compare against a plain tabular baseline
(Logistic Regression, no graph) trained/evaluated on the SAME node
indices for a fair comparison.


In [1]:
import importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
 
# 05_models.py starts with a digit so it can't be `import`-ed normally;
# load it explicitly by file path instead.
_spec = importlib.util.spec_from_file_location("models05", "05_models.py")
_models05 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_models05)
GCN, GAT = _models05.GCN, _models05.GAT
 
DATA_PATH = "Output/graph_data.pt"
OUT_DIR = "Output"
 
torch.manual_seed(42)
np.random.seed(42)
 
data = torch.load(DATA_PATH, weights_only=False)
print(f"Loaded graph: {data}")

Loaded graph: Data(x=[704, 42], edge_index=[2, 10048], y=[704], train_mask=[704], val_mask=[704], test_mask=[704])

MODEL ARCHITECTURES
GCN(
  (conv1): GCNConv(42, 16)
  (conv2): GCNConv(16, 2)
)
GCN trainable parameters: 722

GAT(
  (conv1): GATConv(42, 16, heads=8)
  (conv2): GATConv(128, 2, heads=1)
)
GAT trainable parameters: 6,022

FORWARD PASS SANITY CHECK (untrained weights, just checking shapes)
GCN output shape: torch.Size([704, 2]) (expected [704, 2])
GAT output shape: torch.Size([704, 2]) (expected [704, 2])

Both models produce correctly shaped output. Ready for Phase 6 (training).
Loaded graph: Data(x=[704, 42], edge_index=[2, 10048], y=[704], train_mask=[704], val_mask=[704], test_mask=[704])


Step 1: Class weights for imbalance-aware loss.
Computed ONLY from the training set -- val/test must never
influence how we weight the loss, or we'd be leaking split info
into a training decision.

In [2]:
train_labels = data.y[data.train_mask]
class_counts = torch.bincount(train_labels)
class_weights = 1.0 / class_counts.float()
class_weights = class_weights / class_weights.sum() * len(class_counts)
print(f"\nTrain class counts: {class_counts.tolist()}")
print(f"Class weights (inverse frequency, normalized): {class_weights.tolist()}")
 
 
def evaluate(model, data, mask):
    """Run model in eval mode, return metrics dict for the given mask."""
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        probs = torch.softmax(out, dim=1)[:, 1]  # P(ASD positive)
        preds = out.argmax(dim=1)
 
    y_true = data.y[mask].numpy()
    y_pred = preds[mask].numpy()
    y_prob = probs[mask].numpy()
 
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
 
 
def train_model(model, data, class_weights, lr, weight_decay,
                 max_epochs=300, patience=30, model_name="model"):
    """
    Full training loop with early stopping on validation F1.
    Returns the best model (loaded with best-val-F1 weights) and a
    history dict for plotting.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
 
    best_val_f1 = -1
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_f1": []}
 
    for epoch in range(1, max_epochs + 1):
        # --- train step ---
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        # Loss computed ONLY on train_mask nodes -- this is the line
        # that actually enforces the transductive train/val/test
        # separation described in Phase 4. The model's forward pass
        # still "sees" every node's features via message passing, but
        # gradients only ever flow from train-labeled predictions.
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
 
        # --- validation step ---
        model.eval()
        with torch.no_grad():
            out_val = model(data.x, data.edge_index)
            val_loss = criterion(out_val[data.val_mask], data.y[data.val_mask]).item()
        val_metrics = evaluate(model, data, data.val_mask)
 
        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_metrics["f1"])
 
        # --- early stopping bookkeeping ---
        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
 
        if epoch % 20 == 0 or epoch == 1:
            print(f"[{model_name}] epoch {epoch:3d} | train_loss={loss.item():.4f} "
                  f"| val_loss={val_loss:.4f} | val_f1={val_metrics['f1']:.4f} "
                  f"(best={best_val_f1:.4f})")
 
        if epochs_no_improve >= patience:
            print(f"[{model_name}] Early stopping at epoch {epoch} "
                  f"(no val F1 improvement for {patience} epochs)")
            break
 
    model.load_state_dict(best_state)
    print(f"[{model_name}] Restored best checkpoint (val_f1={best_val_f1:.4f})")
    return model, history


Train class counts: [360, 132]
Class weights (inverse frequency, normalized): [0.5365853905677795, 1.4634146690368652]


 Train GCN

In [3]:
print("\n" + "=" * 60)
print("TRAINING GCN")
print("=" * 60)
gcn = GCN(in_channels=data.num_node_features, hidden_channels=16, out_channels=2, dropout=0.5)
gcn, gcn_history = train_model(gcn, data, class_weights, lr=0.01, weight_decay=5e-4,
                                model_name="GCN")


TRAINING GCN
[GCN] epoch   1 | train_loss=0.6662 | val_loss=0.6369 | val_f1=0.5625 (best=0.5625)
[GCN] epoch  20 | train_loss=0.2223 | val_loss=0.3126 | val_f1=0.7857 (best=0.7857)
[GCN] epoch  40 | train_loss=0.1260 | val_loss=0.1875 | val_f1=0.8667 (best=0.8667)
[GCN] epoch  60 | train_loss=0.1084 | val_loss=0.1662 | val_f1=0.8852 (best=0.8852)
[GCN] Early stopping at epoch 71 (no val F1 improvement for 30 epochs)
[GCN] Restored best checkpoint (val_f1=0.8852)


Train GAT

In [4]:
print("\n" + "=" * 60)
print("TRAINING GAT")
print("=" * 60)
gat = GAT(in_channels=data.num_node_features, hidden_channels=16, out_channels=2,
          heads=8, dropout=0.6)
gat, gat_history = train_model(gat, data, class_weights, lr=0.005, weight_decay=5e-4,
                                model_name="GAT")


TRAINING GAT
[GAT] epoch   1 | train_loss=0.7286 | val_loss=0.6021 | val_f1=0.6374 (best=0.6374)
[GAT] epoch  20 | train_loss=0.2532 | val_loss=0.2758 | val_f1=0.8421 (best=0.8421)
[GAT] epoch  40 | train_loss=0.2171 | val_loss=0.2406 | val_f1=0.8421 (best=0.8421)
[GAT] epoch  60 | train_loss=0.1783 | val_loss=0.2631 | val_f1=0.8421 (best=0.8710)
[GAT] epoch  80 | train_loss=0.2063 | val_loss=0.2005 | val_f1=0.8571 (best=0.8814)
[GAT] epoch 100 | train_loss=0.1826 | val_loss=0.2026 | val_f1=0.8621 (best=0.8814)
[GAT] Early stopping at epoch 107 (no val F1 improvement for 30 epochs)
[GAT] Restored best checkpoint (val_f1=0.8814)


Test set evaluation for both GNNs

In [5]:
print("\n" + "=" * 60)
print("TEST SET RESULTS (best-val-F1 checkpoint)")
print("=" * 60)
gcn_test = evaluate(gcn, data, data.test_mask)
gat_test = evaluate(gat, data, data.test_mask)
 
for name, m in [("GCN", gcn_test), ("GAT", gat_test)]:
    print(f"\n{name}:")
    print(f"  Accuracy:  {m['accuracy']:.4f}")
    print(f"  Precision: {m['precision']:.4f}")
    print(f"  Recall:    {m['recall']:.4f}")
    print(f"  F1:        {m['f1']:.4f}")
    print(f"  ROC-AUC:   {m['auc']:.4f}")
    print(f"  Confusion matrix:\n{m['confusion_matrix']}")


TEST SET RESULTS (best-val-F1 checkpoint)

GCN:
  Accuracy:  0.9057
  Precision: 0.8214
  Recall:    0.8214
  F1:        0.8214
  ROC-AUC:   0.9684
  Confusion matrix:
[[73  5]
 [ 5 23]]

GAT:
  Accuracy:  0.9151
  Precision: 0.8519
  Recall:    0.8214
  F1:        0.8364
  ROC-AUC:   0.9744
  Confusion matrix:
[[74  4]
 [ 5 23]]


Tabular baselines (NO graph at all) on the identical split

In [6]:
print("\n" + "=" * 60)
print("TABULAR BASELINES (Logistic Regression, Random Forest) -- same split")
print("=" * 60)
X_all = data.x.numpy()
y_all = data.y.numpy()
train_mask_np = data.train_mask.numpy()
test_mask_np = data.test_mask.numpy()
 
X_train, y_train = X_all[train_mask_np], y_all[train_mask_np]
X_test, y_test = X_all[test_mask_np], y_all[test_mask_np]
 
def evaluate_sklearn(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_prob),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }
 
logreg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_test = evaluate_sklearn(logreg, X_test, y_test)
 
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_test = evaluate_sklearn(rf, X_test, y_test)
 
for name, m in [("Logistic Regression", logreg_test), ("Random Forest", rf_test)]:
    print(f"\n{name}:")
    print(f"  Accuracy:  {m['accuracy']:.4f}")
    print(f"  Precision: {m['precision']:.4f}")
    print(f"  Recall:    {m['recall']:.4f}")
    print(f"  F1:        {m['f1']:.4f}")
    print(f"  ROC-AUC:   {m['auc']:.4f}")


TABULAR BASELINES (Logistic Regression, Random Forest) -- same split

Logistic Regression:
  Accuracy:  0.9906
  Precision: 0.9655
  Recall:    1.0000
  F1:        0.9825
  ROC-AUC:   0.9991

Random Forest:
  Accuracy:  0.9340
  Precision: 1.0000
  Recall:    0.7500
  F1:        0.8571
  ROC-AUC:   0.9872


Final comparison table

In [7]:
print("\n" + "=" * 60)
print("FINAL COMPARISON TABLE (test set)")
print("=" * 60)
results_table = pd.DataFrame({
    "Logistic Regression": logreg_test,
    "Random Forest": rf_test,
    "GCN": gcn_test,
    "GAT": gat_test,
}).T[["accuracy", "precision", "recall", "f1", "auc"]]
results_table.columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
print(results_table.round(4))
results_table.to_csv(f"{OUT_DIR}/model_comparison.csv")
print(f"\nSaved comparison table to {OUT_DIR}/model_comparison.csv")


FINAL COMPARISON TABLE (test set)
                     Accuracy Precision    Recall        F1   ROC-AUC
Logistic Regression  0.990566  0.965517       1.0  0.982456  0.999084
Random Forest        0.933962       1.0      0.75  0.857143  0.987179
GCN                   0.90566  0.821429  0.821429  0.821429  0.968407
GAT                  0.915094  0.851852  0.821429  0.836364  0.974359

Saved comparison table to Output/model_comparison.csv


Training curve plots

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
 
axes[0].plot(gcn_history["train_loss"], label="GCN train loss", color="#4C72B0")
axes[0].plot(gcn_history["val_loss"], label="GCN val loss", color="#4C72B0", linestyle="--")
axes[0].plot(gat_history["train_loss"], label="GAT train loss", color="#DD8452")
axes[0].plot(gat_history["val_loss"], label="GAT val loss", color="#DD8452", linestyle="--")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training / validation loss")
axes[0].legend(fontsize=8)
 
axes[1].plot(gcn_history["val_f1"], label="GCN val F1", color="#4C72B0")
axes[1].plot(gat_history["val_f1"], label="GAT val F1", color="#DD8452")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation F1")
axes[1].set_title("Validation F1 over training")
axes[1].legend(fontsize=8)
 
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=120)
plt.close()
print(f"Saved training curves to {OUT_DIR}/training_curves.png")

Saved training curves to Output/training_curves.png


Save trained model weights for Phase 7 (interpretability)

In [9]:
torch.save(gcn.state_dict(), f"{OUT_DIR}/gcn_weights.pt")
torch.save(gat.state_dict(), f"{OUT_DIR}/gat_weights.pt")
print(f"\nSaved trained model weights (gcn_weights.pt, gat_weights.pt) for Phase 7")


Saved trained model weights (gcn_weights.pt, gat_weights.pt) for Phase 7
